In [1]:
import os

# Folders to exclude
EXCLUDED_FOLDERS = {'myenv', 'scipy', '__pycache__', '.git','skin-disease-datasaet'}

def print_tree(start_path, indent=""):
    for item in sorted(os.listdir(start_path)):
        item_path = os.path.join(start_path, item)
        if item in EXCLUDED_FOLDERS:
            continue
        if os.path.isdir(item_path):
            print(f"{indent}📁 {item}")
            print_tree(item_path, indent + "    ")
        else:
            print(f"{indent}📄 {item}")

# Replace '.' with your desired root directory
print_tree('.')


📄 llm.ipynb
📄 skin_disease_prediction.ipynb
📄 web_scrapping.ipynb


In [5]:
import pandas as pd
import spacy
import sys
import os

In [6]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from backend.config import MAYO_CSV

In [7]:
df = pd.read_csv(MAYO_CSV)

In [8]:
df.head()

,disease,main_link,Diagnosis_treatment_link,Doctors_departments_link,Overview,Symptoms,When to see a doctor,Causes,Risk factors,Complications,Prevention,Diagnosis,Treatment,Coping and support,Preparing for your appointment,Lifestyle and home remedies
0,Atrial fibrillation,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Atrial fibrillation (AFib) is an irregular and...,Symptoms ofAFibmay include:\nFeelings of a fas...,"If you have symptoms of atrial fibrillation, m...",To understand the causes of atrial fibrillatio...,Things that can increase the risk of atrial fi...,Blood clots are a dangerous complication of at...,Healthy lifestyle choices can reduce the risk ...,You may not know you have atrial fibrillation ...,The goals of atrial fibrillation treatment are...,NaN,If you have an irregular or pounding heartbeat...,Following a heart-healthy lifestyle can help p...
1,Hyperhidrosis,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Hyperhidrosis (hi-pur-hi-DROE-sis) is excessiv...,The main symptom of hyperhidrosis is heavy swe...,Sometimes excessive sweating is a sign of a se...,Sweating is the body's mechanism to cool itsel...,Risk factors for hyperhidrosis include:\nHavin...,Complications of hyperhidrosis include:\nInfec...,NaN,Diagnosing hyperhidrosis may start with your h...,Treating hyperhidrosis may start with treating...,Hyperhidrosis can be the cause of discomfort a...,You may start by seeing your primary care prov...,The following suggestions may help control swe...
2,Bartholin's cyst,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,The Bartholin's (BAHR-toe-linz) glands are loc...,"If you have a small, noninfected Bartholin's c...",Call your doctor if you have a painful lump ne...,Experts believe that the cause of a Bartholin'...,NaN,A Bartholin's cyst or abscess may recur and ag...,There's no way to prevent a Bartholin's cyst. ...,"To diagnose a Bartholin's cyst, your doctor ma...",Often a Bartholin's cyst requires no treatment...,NaN,Your first appointment will likely be with eit...,NaN
3,Infant reflux,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,NaN,Infant reflux is when a baby spits up liquid o...,"Most of the time, infant reflux isn't a cause ...",See a healthcare professional if a baby:\nIsn'...,"In infants, the ring of muscle between the eso...",Infant reflux is common. But some things make ...,Infant reflux usually gets better on its own. ...,NaN,"To diagnose infant reflux, a healthcare profes...","For most babies, making some changes to feedin...",NaN,You may start by seeing your baby's primary he...,To minimize reflux:\nFeed your baby in an upri...
4,Hidradenitis suppurativa,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,https://www.mayoclinic.org/diseases-conditions...,Hidradenitis suppurativa (hi-drad-uh-NIE-tis s...,Hidradenitis suppurativa can affect one or sev...,Early diagnosis of hidradenitis suppurativa is...,Hidradenitis suppurativa develops when hair fo...,Factors that increase your chance of developin...,Persistent and severe hidradenitis suppurativa...,NaN,Hidradenitis suppurativa can be mistaken for p...,"Treatment with medicines, surgery or both can ...",Hidradenitis suppurativa can be a challenge to...,You'll likely first see your primary care prov...,Mild hidradenitis suppurativa can sometimes be...


In [9]:
symptoms = df["Symptoms"][0]

In [12]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(symptoms)

for ent in doc.ents:
    print(ent.text, ent.label_)


c:\Users\harme\AppData\Local\Programs\Python\Python39\lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


AFib ORG
a few minutes to hours TIME
as long as a week DATE
12 months DATE


In [14]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "d4data/biomedical-ner-all"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

entities = ner(symptoms)
for e in entities:
    if e['entity_group'] == 'Sign_symptom':
        print(e['word'], e['entity_group'])


Device set to use cpu


pal Sign_symptom
##pit Sign_symptom
pain Sign_symptom
di Sign_symptom
to Sign_symptom
short Sign_symptom
afibsymptoms Sign_symptom
symptoms Sign_symptom
symptoms Sign_symptom
irregular heartbeat Sign_symptom
symptoms Sign_symptom


In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import HumanMessage
import os

# For Groq
os.environ["OPENAI_API_KEY"] = os.getenv("GROQ_API_KEY") or "your_groq_key"
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

llm = ChatOpenAI(
    model="llama3-70b-8192",  # or llama3-70b-8192 if supported
    temperature=0
)



C:\Users\harme\AppData\Local\Temp\ipykernel_21916\3804315171.py:9: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(


In [19]:
from langchain.prompts import PromptTemplate

prompt_template_name = PromptTemplate(
    input_variables=['symptoms'],
        template="""
Given the following paragraph where a patient is casually describing what they are experiencing, extract only the medical symptoms mentioned. 

Do not include any extra text, disease names, or explanations. Just return a clean, comma-separated list of symptoms extracted from the paragraph. dont event add this text Here is the list of symptoms extracted from the paragraph:

Make it ready for word to word embeddings and matching
Paragraph: {symptoms}
Symptoms (comma-separated):
"""
)


In [20]:
chain = prompt_template_name | llm


In [21]:
response = chain.invoke({"symptoms":symptoms})

In [22]:
response.content

'Feelings of a fast, fluttering or pounding heartbeat, palpitations, Chest pain, Dizziness, Fatigue, Lightheadedness, Reduced ability to exercise, Shortness of breath, Weakness'

In [24]:
import spacy

# Load the biomedical NER model
nlp = spacy.load("en_ner_bc5cdr_md")

def extract_entities(text):
    doc = nlp(text)
    for ent in doc.ents:
        print(f"Text: {ent.text} | Label: {ent.label_}")


In [25]:
extract_entities(symptoms)

Text: pounding | Label: DISEASE
Text: palpitations | Label: DISEASE
Text: Chest pain | Label: DISEASE
Text: Dizziness | Label: DISEASE
Text: Lightheadedness | Label: DISEASE
Text: Shortness of breath | Label: DISEASE
Text: atrial fibrillation | Label: DISEASE
Text: AFib | Label: DISEASE
Text: Atrial fibrillation | Label: DISEASE
Text: atrial fibrillation | Label: DISEASE
Text: atrial fibrillation | Label: DISEASE
